### Endpoint for SAFE conversion process in DPR service that takes as input a path to a SAFE product in a bucket, calls CPM to transform the product, then returns the path to the converted product.

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-675


In [1]:
# Init environment before running a demo notebook.
import os
from resources.utils import *  
from resources.dask_utils import *

init_demo()
await init_dask_cluster_cpm(scale=1)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

Auxip service: http://rs-server-adgs:8000/auxip
PRIP service: http://rs-server-prip:8000/prip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
OSAM service: http://rs-server-osam:8000
Initializing dask-cpm cluster. This can take some time...
Dask version used: 2026.1.2
Connecting to dask gateway for 'dask-cpm.alin.local': http://dask-cpm:8000 ...
Get existing dask cluster: 'e5b22b9f88db406680b924d76e3f0491'
Dask cluster name: e5b22b9f88db406680b924d76e3f0491
Dask dashboard for 'dask-cpm.alin.local': http://localhost:8000/dask/cpm/clusters/e5b22b9f88db406680b924d76e3f0491/status
Dask workers for 'dask-cpm.alin.local' are up: 1/1
Dask cluster initialized. Waiting for STOP signal to exit...


None

In [2]:
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf

from rs_client.rs_client import RsClient
rs_server_href = os.getenv("RSPY_WEBSITE")
rs_server_api_key = os.environ.get("RSPY_APIKEY")
generic_client = RsClient(rs_server_href, rs_server_api_key, OWNER_ID, None)

In [3]:
import os
import s3fs

safe_name = "S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D.SAFE"

fs = s3fs.S3FileSystem(
    key=os.environ["S3_ACCESSKEY"],
    secret=os.environ["S3_SECRETKEY"],
    client_kwargs={
        "endpoint_url": os.environ["S3_ENDPOINT"],
        "region_name": os.environ["S3_REGION"],
    },
)

fs.put(
    f"../../../CDSE/{safe_name}",
    f"rs-dev-cluster-temp/conversion/{safe_name}",
    recursive=True,
)



[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [4]:
safe_name = "S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D.SAFE"

result = submit_dask_cpm_task(f"""
def task():
    import os
    import shutil
    import fsspec
    from pathlib import Path
    from eopf.config import EOConfiguration
    from eopf.store.convert import convert

    safe_name = "{safe_name}"
    safe_s3 = f"rs-dev-cluster-temp/conversion/{{safe_name}}"
    local_safe = f"/tmp/{{safe_name}}"
    target = "/tmp/zarr-output-from-s3"

    fs = fsspec.filesystem(
        "s3",
        key=os.environ["S3_ACCESSKEY"],
        secret=os.environ["S3_SECRETKEY"],
        client_kwargs={{
            "endpoint_url": os.environ["S3_ENDPOINT"],
            "region_name": os.environ["S3_REGION"],
        }},
    )

    shutil.rmtree(local_safe, ignore_errors=True)
    shutil.rmtree(target, ignore_errors=True)

    fs.get(safe_s3, local_safe, recursive=True)

    cfg = EOConfiguration()
    cfg["dask_context__cluster_config__memory_limit"] = "24GiB"
    cfg["dask_context__cluster_config__n_workers"] = 1
    cfg["dask_context__cluster_config__threads_per_worker"] = 1
    cfg["dask_context__cluster_config__processes"] = True

    _, product_name = convert(local_safe, target, target_format="zarr")
    zarr_path = f"{{target}}/{{product_name}}.zarr"

    return {{
        "product_name": product_name,
        "local_safe_exists": Path(local_safe).exists(),
        "zarr_exists": Path(zarr_path).exists(),
        "zarr_path": zarr_path,
    }}
""")

result


{'product_name': 'S01SIWSLC_20240416T171518_0026_A015_T14F',
 'local_safe_exists': True,
 'zarr_exists': True,
 'zarr_path': '/tmp/zarr-output-from-s3/S01SIWSLC_20240416T171518_0026_A015_T14F.zarr'}

In [5]:
legacy_products = [
    #"S3A_OL_1_EFR____20240626T125108_20240626T125215_20240626T141905_0067_114_052_3780_PS1_O_NR_004.SEN3",
    #"S3A_OL_2_LFR____20240430T083943_20240430T084243_20240501T092030_0179_112_007_2160_PS1_O_NT_002.SEN3.zip",
    #"S3B_OL_2_LFR____20240430T130043_20240430T130343_20240501T004249_0179_092_252_1980_PS2_O_NT_002.SEN3",
    #"S3B_SY_2_V10____20240611T000000_20240620T235959_20240622T121945_EUROPE____________PS2_O_ST_002.SEN3",
    "S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D.SAFE",
]

payloads = [
    {
        "input_safe_path": f"s3://rs-dev-cluster-temp/conversion/{legacy_product}",
        "output_zarr_dir_path": "s3://rs-dev-cluster-temp/conversion/",
    }
    for legacy_product in legacy_products
]

In [8]:
dpr_client = generic_client.get_dpr_client()

# Launch all conversion jobs - "/dpr/processes/{resource}/execution" endpoint with "conv_safe_zarr" as resource
job_status_list = [dpr_client.run_conv_safe_zarr(payload, cluster_info_eopf) for payload in payloads]

results = [dpr_client.wait_for_job(job_status, logger=None, job_name="Conversion Processor") for job_status in job_status_list]
results

RuntimeError: Conversion Processor job 'a3f9ac5b-9cfc-40be-a02f-329f56d7a284': FAILED

In [ ]:
responses = [requests.get(f"{dpr_client.href_service}/dpr/jobs/{job_status['jobID']}").json() for job_status in job_status_list]
responses

In [ ]:
shutdown = False
if shutdown:
    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

    # Close the python objects
    close_dask_clusters()

In [ ]:
from distributed import get_worker

def simple_task():
    worker = get_worker()
    return {
        "worker": worker.address,
        "message": "task ran on dask-cpm",
    }

future = dask_client_eopf.submit(simple_task)
future.result()
